## 1. ONNX Export 환경 및 학습 모델 준비

1. 필수 라이브러리 불러오기

In [5]:
%pip install -q onnx onnxruntime

In [6]:
%pip install -q --upgrade onnx onnxscript onnxruntime

In [7]:
%pip install -q \
    "numpy==2.2.6" \
    "numba==0.61.2" \
    "anomalib==2.6.2" \
    jedi

In [8]:
%pip install --force-reinstall --no-cache-dir "numpy==2.1.3"

In [9]:
from pathlib import Path
import importlib.util

import torch

from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.deploy import ExportType

2. ONNX 관련 패키지 설치 여부 확인

In [10]:
packages = [
    "onnx",
    "onnxruntime",
]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package:12s}: {installed}")

3. 현재 실행 환경 확인

In [11]:
import anomalib

print("PyTorch :", torch.__version__)
print("Anomalib:", anomalib.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

4. 저장된 Checkpoint 확인

In [12]:
checkpoint_path = Path(
    "./models/patchcore_bottle.ckpt"
)

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint path  :", checkpoint_path)

5. MVTec AD DataModule 구성

In [13]:
datamodule = MVTecAD(
    root="./datasets/MVTecAD",
    category="bottle",
    train_batch_size=32,
    eval_batch_size=32,
)

datamodule.prepare_data()
datamodule.setup()

6. 학습과 동일한 PatchCore 구조 생성

In [14]:
pre_processor = Patchcore.configure_pre_processor(
    image_size=(256, 256),
    center_crop_size=(256, 256),
)

model = Patchcore(
    backbone="wide_resnet50_2",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
    pre_processor=pre_processor,
)

7. Export용 Engine 생성

In [15]:
engine = Engine(
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)

## 2. PatchCore 배포 구조 분리 및 Feature Extractor Export

1. 학습된 PatchCore Checkpoint 불러오기

In [16]:
trained_model = Patchcore.load_from_checkpoint(
    str(checkpoint_path),
    weights_only=False,
)

trained_model.eval()

print("Checkpoint loaded.")
print(
    "Memory bank shape:",
    trained_model.model.memory_bank.shape,
)

2. Memory Bank 별도 저장

In [ ]:
import numpy as np

memory_bank = (
    trained_model.model
    .memory_bank
    .detach()
    .cpu()
    .numpy()
)

memory_bank_path = Path(
    "./models/patchcore_memory_bank.npy"
)

np.save(
    memory_bank_path,
    memory_bank,
)

print("Memory bank shape:", memory_bank.shape)
print("Saved to         :", memory_bank_path)

3. Feature Extractor Wrapper 정의

In [ ]:
import torch
import torch.nn.functional as F


class PatchcoreFeatureExtractor(torch.nn.Module):
    def __init__(self, patchcore_model):
        super().__init__()

        self.feature_extractor = (
            patchcore_model.feature_extractor
        )

        self.feature_pooler = (
            patchcore_model.feature_pooler
        )

        self.layers = list(
            patchcore_model.layers
        )

    def forward(self, x):
        features = self.feature_extractor(x)

        features = {
            layer: self.feature_pooler(
                features[layer]
            )
            for layer in self.layers
        }

        embedding = features[
            self.layers[0]
        ]

        for layer in self.layers[1:]:
            layer_embedding = features[layer]

            layer_embedding = F.interpolate(
                layer_embedding,
                size=embedding.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            embedding = torch.cat(
                (
                    embedding,
                    layer_embedding,
                ),
                dim=1,
            )

        return embedding

4. Feature Extractor 모델 생성

In [ ]:
feature_model = PatchcoreFeatureExtractor(
    trained_model.model
)

feature_model.eval()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

feature_model = feature_model.to(device)

print("Device:", device)

5. Dummy Input 생성

In [ ]:
dummy_input = torch.randn(
    1,
    3,
    256,
    256,
    device=device,
)

print("Input shape:", dummy_input.shape)

6. PyTorch Feature Embedding 출력 확인

In [ ]:
with torch.no_grad():
    embedding = feature_model(
        dummy_input
    )

print(
    "Embedding shape:",
    embedding.shape,
)

## 3. PatchCore Feature Extractor ONNX Export

1. ONNX 저장 폴더 생성

In [ ]:
onnx_export_dir = Path(
    "./models/onnx"
)

onnx_export_dir.mkdir(
    parents=True,
    exist_ok=True,
)

feature_onnx_path = (
    onnx_export_dir
    / "patchcore_feature_extractor.onnx"
)

print(
    "ONNX output:",
    feature_onnx_path,
)

2. Feature Extractor ONNX Export

In [ ]:
torch.onnx.export(
    feature_model,
    (dummy_input,),
    str(feature_onnx_path),

    input_names=[
        "input"
    ],

    output_names=[
        "embedding"
    ],

    dynamo=True,
)

print(
    "ONNX export completed:",
    feature_onnx_path,
)

3. ONNX 파일 생성 확인

In [ ]:
print(
    "Exists:",
    feature_onnx_path.exists(),
)

if feature_onnx_path.exists():
    size_mb = (
        feature_onnx_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"ONNX size: {size_mb:.2f} MB"
    )

## 4. ONNX 모델 구조 검증

1. ONNX 모델 Load

In [ ]:
import onnx

onnx_model = onnx.load(
    str(feature_onnx_path)
)

print(
    "ONNX model loaded."
)

2. ONNX Checker 실행

In [ ]:
onnx.checker.check_model(
    onnx_model
)

print(
    "ONNX model validation passed."
)

3. ONNX Input 확인

In [ ]:
print("Inputs:")

for model_input in onnx_model.graph.input:
    print(
        "-",
        model_input.name,
    )

4. ONNX Output 확인

In [ ]:
print("Outputs:")

for model_output in onnx_model.graph.output:
    print(
        "-",
        model_output.name,
    )

5. ONNX Operator 확인

In [ ]:
from collections import Counter

op_counts = Counter(
    node.op_type
    for node in onnx_model.graph.node
)

for op_name, count in op_counts.most_common():
    print(
        f"{op_name:20s}: {count}"
    )

## 2. PatchCore 배포 구조 분리 및 Feature Extractor 준비

1. 학습된 PatchCore checkpoint 불러오기

In [ ]:
trained_model = Patchcore.load_from_checkpoint(
    str(checkpoint_path),
    weights_only=False,
)

trained_model.eval()

print("Checkpoint loaded.")
print(
    "Memory bank shape:",
    trained_model.model.memory_bank.shape,
)